In [ ]:
"""
CNN + Temporal Convolutional Network (TCN)
核心：膨胀卷积，全并行，通过指数增长的 dilation 覆盖长距离依赖
"""
import torch
import torch.nn as nn
from cnn_frontend import CNNFrontend, ClassifierHead

In [ ]:
class TCNBlock(nn.Module):
    """
    单个 TCN 残差块：两层膨胀卷积 + 残差连接
    使用对称 padding（非因果），适合分类任务
    """
    def __init__(self, channels, kernel_size, dilation, dropout=0.2):
        super().__init__()
        # 对称 padding 使输出长度 = 输入长度
        padding = dilation * (kernel_size - 1) // 2

        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size,
                      dilation=dilation, padding=padding),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Conv1d(channels, channels, kernel_size, 
                      dilation=dilation, padding=padding),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x: (batch, channels, seq_len)
        return x + self.net(x) # 残差连接（通道数不变，无需投影）

# TCN 残差块（TCNBlock）详解

## 一、整体架构图

```
输入 x ─────────────────────────────┐
  │                                  │ (残差连接/跳跃连接)
  ▼                                  │
┌─────────────────────────────┐      │
│  膨胀因果卷积 Conv1d         │      │
│  BatchNorm1d                │      │
│  GELU 激活                  │      │  第一层
│  Dropout                    │      │
├─────────────────────────────┤      │
│  膨胀因果卷积 Conv1d         │      │
│  BatchNorm1d                │      │
│  GELU 激活                  │      │  第二层
│  Dropout                    │      │
└──────────────┬──────────────┘      │
               │                     │
               ▼                     │
             (+) ◄───────────────────┘  逐元素相加
               │
               ▼
            输出 (与输入shape完全相同)
```

---

## 二、逐行详解

### 1. 对称 Padding 计算

```python
padding = dilation * (kernel_size - 1) // 2
```

**目的：保证卷积后序列长度不变**（输出长度 = 输入长度）

Conv1d 输出长度公式：

$$L_{out} = \frac{L_{in} + 2 \times \text{padding} - \text{dilation} \times (\text{kernel\_size} - 1) - 1}{1} + 1$$

代入 `padding = dilation × (kernel_size - 1) // 2`：

$$L_{out} = L_{in} + 2 \times \frac{d(k-1)}{2} - d(k-1) = L_{in}$$

**具体例子：**

| kernel_size | dilation | 有效感受野 | padding |
|:-----------:|:--------:|:---------:|:-------:|
| 3 | 1 | 3 | 1 |
| 3 | 2 | 5 | 2 |
| 3 | 4 | 9 | 4 |
| 3 | 8 | 17 | 8 |

---

### 2. 膨胀卷积（Dilated Convolution）—— TCN 的核心

```python
nn.Conv1d(channels, channels, kernel_size, dilation=dilation, padding=padding)
```

**普通卷积 vs 膨胀卷积的直观对比：**

```
普通卷积 (dilation=1, kernel_size=3):
输入:  [a] [b] [c] [d] [e] [f] [g]
        \___|___/
        连续3个元素，感受野 = 3

膨胀卷积 (dilation=2, kernel_size=3):
输入:  [a] [b] [c] [d] [e] [f] [g]
        \___skip___\___skip___/
        间隔2采样，感受野 = 5

膨胀卷积 (dilation=4, kernel_size=3):
输入:  [a] [b] [c] [d] [e] [f] [g] [h] [i]
        \____skip×4____\____skip×4____/
        间隔4采样，感受野 = 9
```

**关键优势：不增加参数量，指数级扩大感受野**

---

### 3. 对称 Padding vs 因果 Padding

```
因果 padding（用于生成/预测）：只在左侧填充，不看未来
────────────────────────────────────────────
  padding全在左边
  [0][0][0][0] [x1][x2][x3][x4][x5]
  → 时刻 t 的输出只依赖 ≤t 的输入

对称 padding（本代码，用于分类）：两侧填充，可以看未来
────────────────────────────────────────────
  [0][0] [x1][x2][x3][x4][x5] [0][0]
  → 时刻 t 的输出依赖前后文，信息更充分
```

> **分类任务已经拿到了完整序列，没必要限制"不看未来"，对称 padding 效果更好。**

---

### 4. 每层组件的作用

```python
self.net = nn.Sequential(
    nn.Conv1d(...),          # ① 提取时序特征（膨胀卷积）
    nn.BatchNorm1d(channels),# ② 归一化，稳定训练
    nn.GELU(),               # ③ 非线性激活
    nn.Dropout(dropout),     # ④ 正则化，防过拟合
    # ---- 重复一次 ----
    nn.Conv1d(...),          
    nn.BatchNorm1d(channels),
    nn.GELU(),               
    nn.Dropout(dropout),     
)
```

**为什么用 GELU 而非 ReLU？**
```
ReLU:  max(0, x)         → 硬截断，负值梯度为0
GELU:  x · Φ(x)          → 平滑门控，Transformer 标配

        │  GELU        ReLU
     1  │    ╱──      ╱──
        │  ╱        ╱
     0  │╱────    ╱────
        │  ~     │
    -1  │        │
        └──────────────
```

---

### 5. 残差连接

```python
def forward(self, x):
    return x + self.net(x)  # 恒等映射 + 变换
```

**为什么有效：**

```
没有残差连接：
  梯度需要穿过所有层 → 深层网络梯度消失

有残差连接：
  梯度有"高速公路"直接回传
  
  ∂Loss/∂x = ∂Loss/∂out × (1 + ∂net(x)/∂x)
                              ↑
                          这个 1 保证梯度至少为 1
```

**前提条件：输入输出 shape 必须完全相同**
- 通道数：`channels → channels`（不变）
- 序列长度：对称 padding 保证不变
- 所以**无需 1×1 投影层**

---

## 三、多层堆叠时的感受野增长

TCN 通常按 `dilation = 1, 2, 4, 8, ...` 堆叠多个 TCNBlock：

```python
# 典型用法
blocks = [TCNBlock(ch, kernel_size=3, dilation=2**i) for i in range(4)]
```

```
Block 0 (d=1):  感受野 =  5   (两层 conv，每层 k=3,d=1)
Block 1 (d=2):  感受野 =  13
Block 2 (d=4):  感受野 =  29
Block 3 (d=8):  感受野 =  61

感受野呈指数增长！4个 block 就能覆盖长度 61 的序列
而普通 CNN 需要 ~30 层才能达到同样的感受野
```

**感受野公式（每个 block 含 2 层 conv）：**

$$RF = 1 + 2(k-1) \sum_{i=0}^{n-1} d_i$$

---

## 四、数据流维度追踪

```python
# 假设 batch=32, channels=128, seq_len=1000, kernel_size=3, dilation=4
x: (32, 128, 1000)

# padding = 4 * (3-1) // 2 = 4
# Conv1d: padding=4 两侧各填充4个零

# 第一层 Conv1d
→ (32, 128, 1000)  # padding 保证长度不变
# BatchNorm1d
→ (32, 128, 1000)  # 对每个通道归一化
# GELU + Dropout
→ (32, 128, 1000)  # 逐元素操作，shape不变

# 第二层（同上）
→ (32, 128, 1000)

# 残差相加
x + net(x) → (32, 128, 1000)  # 完全相同的 shape
```

---

## 五、总结：TCNBlock 的设计哲学

| 设计选择 | 原因 |
|---------|------|
| **膨胀卷积** | 指数级扩大感受野，参数量不增 |
| **对称 padding** | 分类任务无需因果约束，利用全部上下文 |
| **残差连接** | 缓解梯度消失，允许堆叠更深 |
| **BatchNorm** | 稳定训练，加速收敛 |
| **两层卷积/块** | 增加非线性表达能力，经验上效果好 |
| **通道数不变** | 简化残差连接，无需投影层 |

> **一句话概括：TCNBlock 用两层膨胀卷积提取多尺度时序特征，通过对称 padding 保持序列长度不变，通过残差连接保证深层网络的可训练性。**

In [ ]:
class CNN_TCN(nn.Module):
    """
    架构: CNN(1024→128) → TCN(6层膨胀卷积) → GAP → Classifier
    输入: (batch, 1024, 2)

    感受野计算 (kernel=3, dilations=[1,2,4,8,16,32]):
      每层每侧扩展 dilation*(k-1)/2 = dilation
      总感受野 = 1 + 2*Σ(dilation_i) = 1 + 2*(1+2+4+8+16+32) = 127
      → 几乎覆盖整个 128 长度的序列 ✓
    """
    def __init__(self, num_classes=9, channels=128, num_layers=6,
                 kernel_size=3, dropout=0.2):
        super().__init__()

        self.cnn = CNNFrontend(in_channels=2, out_channels=channels)

        tcn_blocks = []
        for i in range(num_layers):
            # 无论 dilation 取什么值，TCN Block 的输入输出形状恒相同
            dilation = 2 ** i # dilation= 2^0, 2^1, 2^2, 2^3, 2^4, 2^5
            tcn_blocks.append(TCNBlock(channels, kernel_size, dilation, dropout))
        self.tcn = nn.Sequential(*tcn_blocks) # 加 * 为解包，因为Sequential不接受列表，因此需要将列表tcn_blocks解包，以确保传入独立参数

        self.final_norm = nn.BatchNorm1d(channels)
        self.classifier = ClassifierHead(channels, num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        # x: (batch, 1024, 2)
        x = x.permute(0, 2, 1)  # → (batch, 2, 1024)
        x = self.cnn(x)         # → (batch, 128, 128)
        x = self.tcn(x)         # → (batch, 128, 128)
        x = self.final_norm(x)
        x = x.mean(dim=2)       # Global Average Pooling → (batch, 128)
        return self.classifier(x)

def build_model(classes=9):
    return CNN_TCN(num_classes=classes)

if __name__ == '__main__':
    model = build_model(9)
    x = torch.randn(4, 1024, 2)
    out = model(x)
    print(f"Input: {x.shape} → Output: {out.shape}")
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {total:,}")

# 膨胀卷积感受野计算详解

---

## 一、先澄清"间隔"的含义

### dilation=4 的含义

**dilation=4 表示相邻采样点的「距离」是 4，不是「跳过」4 个。**

```
位置编号:  a  b  c  d  e  f  g  h  i
索引:      0  1  2  3  4  5  6  7  8

dilation=4, kernel_size=3 采样的三个位置:
           ↑           ↑           ↑
          [0]         [4]         [8]
           a           e           i

a 到 e 的距离 = 4（跳过 b,c,d 共 3 个）
e 到 i 的距离 = 4（跳过 f,g,h 共 3 个）
```

### 类比：你家到学校

```
你家 → [1号楼] → [2号楼] → [3号楼] → 学校
  0        1        2        3       4

"距离 4 栋楼" = 中间经过 3 栋楼
            不是跳过 4 栋楼到第 5 栋
```

### 通用规则

$$\text{跳过的元素数} = \text{dilation} - 1$$

| dilation | 跳过几个 | 示例（从 a 出发） |
|:--------:|:-------:|:-----------------|
| 1 | 0 | a → **b**（紧挨着） |
| 2 | 1 | a → ~~b~~ → **c** |
| 4 | 3 | a → ~~b~~ → ~~c~~ → ~~d~~ → **e** |
| 8 | 7 | a → ~~b~~ → ... → ~~h~~ → **i** |

---

## 二、单层的感受野

### kernel_size=3, dilation=d 的一层卷积

每层卷积取 **3 个点**，间距为 d：

$$\text{单层覆盖范围} = d \times (k-1) + 1 = 2d + 1$$

| dilation | 采样位置 | 覆盖范围 |
|:--------:|:--------|:--------:|
| d=1 | `[0, 1, 2]` | **3** |
| d=2 | `[0, 2, 4]` | **5** |
| d=4 | `[0, 4, 8]` | **9** |
| d=8 | `[0, 8, 16]` | **17** |

---

## 三、逐层堆叠感受野计算

### 核心思路

每一层在上一层的基础上，**向两侧各扩展 dilation 个位置**。

$$\text{每层每侧扩展} = d \times \frac{k-1}{2} = d \times 1 = d$$

### 逐层推导

```
第 0 层 (d=1):  每侧扩展 1
第 1 层 (d=2):  每侧扩展 2
第 2 层 (d=4):  每侧扩展 4
第 3 层 (d=8):  每侧扩展 8
第 4 层 (d=16): 每侧扩展 16
第 5 层 (d=32): 每侧扩展 32
```

$$\text{总感受野} = 1 + 2 \times \sum_{i=0}^{5} d_i = 1 + 2 \times (1+2+4+8+16+32) = 1 + 2 \times 63 = 127$$

---

## 四、用图从底部向上推导（最直观）

### 只画 3 层（d=1, 2, 4），从底层输入往上看

```
Layer 2 输出:  计算 1 个输出点 Y
(d=4)                    │
               ┌─────────┼─────────┐
               ↓         ↓         ↓
Layer 1 输出:  ●    ·  ·  ●  ·  ·  ●         取 3 个点，间距 4
(d=2)          │         │         │
          ┌────┼────┐ ┌──┼──┐ ┌────┼────┐
          ↓    ↓    ↓ ↓  ↓  ↓ ↓    ↓    ↓
Layer 0:  ●  · ●  · ● ● · ● ● ·  ●  · ●    每个 ● 又取 3 个点，间距 2
(d=1)     │   │   │ │   │ │   │   │   │
          ↓   ↓   ↓ ↓   ↓ ↓   ↓   ↓   ↓
输入:   a b c d e f g h i j k l m              
        ↑                               ↑
       [0]                             [12]
        └──── 感受野 = 13 个位置 ────────┘
```

验证：$1 + 2(1+2+4) = 1 + 14 = 15$

等一下，让我更仔细地画：

### 精确推导：3 层 (d=1, d=2, d=4)

**从上往下追踪，看一个输出点依赖哪些输入点。**

#### Layer 2 (d=4)：输出 Y 依赖 Layer 1 的 3 个位置

```
Layer 1 的位置:  ... 3  4  5  6  7  8  9  10 11 ...
                     ↑           ↑           ↑
Y 依赖:            [中心-4]    [中心]     [中心+4]
假设中心=7:          [3]        [7]        [11]
```

#### Layer 1 (d=2)：这 3 个位置各依赖 Layer 0 的 3 个位置

```
位置 3 依赖 Layer 0 的: [3-2]=1, [3]=3, [3+2]=5
位置 7 依赖 Layer 0 的: [7-2]=5, [7]=7, [7+2]=9
位置 11 依赖 Layer 0 的: [11-2]=9, [11]=11, [11+2]=13

合并去重: {1, 3, 5, 7, 9, 11, 13}
```

#### Layer 0 (d=1)：这 7 个位置各依赖输入的 3 个位置

```
位置 1 依赖输入的: 0, 1, 2
位置 3 依赖输入的: 2, 3, 4
位置 5 依赖输入的: 4, 5, 6
位置 7 依赖输入的: 6, 7, 8
位置 9 依赖输入的: 8, 9, 10
位置 11 依赖输入的: 10, 11, 12
位置 13 依赖输入的: 12, 13, 14

合并去重: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14}
```

$$\text{感受野} = 15 \text{ 个连续位置}$$

$$\text{验证}: 1 + 2 \times (1 + 2 + 4) = 1 + 14 = 15 \quad ✅$$

---

## 五、公式推导

### 递推法

设第 $i$ 层后的感受野为 $R_i$：

$$R_0 = 1 + d_0 \times (k-1) = 1 + 1 \times 2 = 3$$

每增加一层，感受野增长：

$$R_{i} = R_{i-1} + d_i \times (k-1) = R_{i-1} + 2d_i$$

| 层 | dilation $d_i$ | 增长 $2d_i$ | 感受野 $R_i$ |
|:--:|:--:|:--:|:--:|
| 初始 | — | — | 1 |
| 0 | 1 | +2 | **3** |
| 1 | 2 | +4 | **7** |
| 2 | 4 | +8 | **15** |
| 3 | 8 | +16 | **31** |
| 4 | 16 | +32 | **63** |
| 5 | 32 | +64 | **127** |

### 通项公式

$$\boxed{R = 1 + (k-1) \sum_{i=0}^{L-1} d_i = 1 + 2\sum_{i=0}^{L-1} 2^i = 1 + 2(2^L - 1) = 2^{L+1} - 1}$$

代入 $L=6$：

$$R = 2^7 - 1 = 127$$

> 几乎覆盖整个长度 128 的序列 ✓

---

## 六、感受野增长对比图

```
层数          感受野可视化（序列长度128）

0层(d=1):     [███]                                          RF=3
              只看 3 个邻居

1层(d=2):     [███████]                                      RF=7
              
2层(d=4):     [███████████████]                               RF=15

3层(d=8):     [███████████████████████████████]                RF=31

4层(d=16):    [███████████████████████████████████████████...] RF=63

5层(d=32):    [████████████████████████████████████████████... RF=127
               ←──────── 几乎覆盖整个 128 ─────────────────→
```

---

## 七、一句话总结

| 问题 | 答案 |
|:-----|:-----|
| dilation=4 的"间隔" | **距离**为 4，跳过 3 个元素（不是跳过 4 个） |
| 感受野公式 | $R = 1 + 2\sum d_i$（kernel_size=3 时） |
| 6 层 TCN 感受野 | $1 + 2(1+2+4+8+16+32) = 127$ |

In [ ]:
"""
CNN + Transformer Encoder
核心：Multi-Head Self-Attention，全并行，全局关系建模
"""
import math
import torch
import torch.nn as nn
from cnn_frontend import CNNFrontend, ClassifierHead

In [ ]:
class PositionalEncoding(nn.Module):
    """正弦位置编码：给序列中每个位置赋予唯一的位置信息"""

    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# 正弦位置编码（Sinusoidal Positional Encoding）详解

这是 **Transformer** 中的经典组件，源自论文 *"Attention Is All You Need"*。

## 核心问题：为什么需要位置编码？

Transformer 的自注意力机制是**置换不变的**（permutation invariant），即打乱输入顺序，输出不变。但语言是有顺序的（"我打你" ≠ "你打我"），所以需要**手动注入位置信息**。

---

## 逐行代码解析

### 1. 初始化：构造位置编码矩阵

```python
pe = torch.zeros(max_len, d_model)  # 创建全零矩阵，形状 (512, d_model)
```

> 预先为最多 512 个位置，每个位置生成一个 `d_model` 维的编码向量。

---

### 2. 构造 `position` 和 `div_term`

```python
position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
# shape: (max_len, 1) = (512, 1)
# [[0.], [1.], [2.], ..., [511.]]
```

```python
div_term = torch.exp(
    torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
)
# shape: (d_model/2,)
```

`div_term` 的数学本质是：

$$\text{div\_term}_i = \frac{1}{10000^{2i/d_{model}}} = e^{-2i \cdot \ln(10000)/d_{model}}$$

> 这是一个**几何递减序列**，从 1 逐渐衰减到接近 0，控制不同维度的**波长从短到长**。

---

### 3. 填充正弦和余弦值

```python
pe[:, 0::2] = torch.sin(position * div_term)  # 偶数维度用 sin
pe[:, 1::2] = torch.cos(position * div_term)  # 奇数维度用 cos
```

对应论文中的公式：

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

**直观理解**：把每个位置编码成不同频率的正弦/余弦波的叠加，类似于"二进制钟表"：

```
维度0,1 (高频): 快速变化 → 区分相邻位置
维度2,3 (中频): 中速变化 → 区分中等距离
  ...
维度d-2,d-1 (低频): 缓慢变化 → 区分远距离位置
```

---

### 4. 注册为 buffer

```python
pe = pe.unsqueeze(0)  # (1, max_len, d_model)，增加batch维度
self.register_buffer('pe', pe)
```

> `register_buffer`：**不参与梯度更新**，但会随模型保存/加载/移动设备（`.cuda()`）。

---

### 5. 前向传播

```python
def forward(self, x):
    # x: (batch, seq_len, d_model) — 词嵌入向量
    x = x + self.pe[:, :x.size(1), :]  # 截取前 seq_len 个位置编码，直接相加
    return self.dropout(x)
```

**关键操作**：词嵌入 + 位置编码 → **逐元素相加**（不是拼接）。

---

## 可视化理解

假设 `d_model=8`，前 10 个位置的编码矩阵热力图：

```
pos\dim   0(sin) 1(cos) 2(sin) 3(cos) 4(sin) 5(cos) 6(sin) 7(cos)
  0       0.00   1.00   0.00   1.00   0.00   1.00   0.00   1.00
  1       0.84   0.54   0.10   0.99   0.01   1.00   0.00   1.00
  2       0.91  -0.42   0.20   0.98   0.02   1.00   0.00   1.00
  3       0.14  -0.99   0.30   0.95   0.03   1.00   0.00   1.00
  ...     ↑高频振荡      ↑中频变化        ↑低频，几乎不变
```

---

## 总结流程图

```
输入序列 "I love NLP"
     ↓
词嵌入 Embedding → (batch, 3, d_model)
     ↓
  ╔══════════════════════╗
  ║  + 位置编码 PE       ║  ← 让模型知道 "I"在位置0，"love"在位置1...
  ╚══════════════════════╝
     ↓
  Dropout
     ↓
送入 Transformer Encoder
```

**设计优点**：
- ✅ **无需学习**，确定性计算
- ✅ 可以**外推**到训练时未见过的更长序列（理论上）
- ✅ 相对位置可通过线性变换表示：$PE_{pos+k}$ 可以表示为 $PE_{pos}$ 的线性函数

# 重新讲解：正弦位置编码

## 第一步：目标是什么

Transformer 无法感知顺序，所以我们需要给每个位置生成一个 **d_model 维的向量**，作为该位置的"指纹"，加到词嵌入上。

```
位置0 → [0.xx, 0.xx, 0.xx, ..., 0.xx]   ← d_model 个数
位置1 → [0.xx, 0.xx, 0.xx, ..., 0.xx]
位置2 → [0.xx, 0.xx, 0.xx, ..., 0.xx]
...
```

要求：
- 每个位置的向量**必须不同**（否则无法区分）
- 值的范围要**有界**（不能爆炸）
- **相近位置**的向量应该**相似**

---

## 第二步：用什么方法生成？—— 二进制的启发

先想一个你已经熟悉的东西：**二进制计数**

```
位置0:  0 0 0 0
位置1:  0 0 0 1    ← 最右列：每1个数翻转一次（最快）
位置2:  0 0 1 0    ← 倒数第2列：每2个数翻转一次
位置3:  0 0 1 1
位置4:  0 1 0 0    ← 倒数第3列：每4个数翻转一次
位置5:  0 1 0 1
位置6:  0 1 1 0
位置7:  0 1 1 1
```

观察每一**列**：
- **右边的列**变化很快（高频），区分相邻位置
- **左边的列**变化很慢（低频），区分远距离位置
- 每一列的"翻转周期"按 **1, 2, 4, 8...** 几何递增

这就够用了！每个位置有唯一编码，且不同列承担不同粒度的区分任务。

**正弦位置编码就是这个思想的连续平滑版本**：把 0/1 的跳变换成 sin/cos 的平滑波动。

---

## 第三步：公式

论文给出的公式：

$$PE(pos, 2i) = \sin\!\left(\frac{pos}{10000^{\,2i/d_{model}}}\right)$$

$$PE(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{\,2i/d_{model}}}\right)$$

- `pos`：位置（第几个 token，0, 1, 2, ...）
- `i`：维度编号（第几个维度，0, 1, 2, ...）
- `d_model`：向量总维度

关键在分母 $10000^{2i/d_{model}}$，它就是代码里叫 **`div_term`** 的东西——全称是 **divisor term（除数项）**。

| 维度 i | 分母 $10000^{2i/d_{model}}$ | 分母大小 | sin 变化快慢 |
|--------|---------------------------|---------|------------|
| i=0（前面的维度）| $10000^0 = 1$ | 小 | **变化快**（高频）|
| i 中间 | 中等 | 中等 | 中等 |
| i 最大（后面的维度）| $10000^1 = 10000$ | 大 | **变化慢**（低频）|

> **分母越大 → `pos/分母` 变化越慢 → sin 振荡越慢**

这和二进制完全对应：
```
二进制：右边列周期短(快)，左边列周期长(慢)，周期按几何递增(×2)
正弦编码：前面维度频率高(快)，后面维度频率低(慢)，频率按几何递减(÷某个数)
```

---

## 第四步：代码如何实现这个公式

现在逐行看代码，每一行都在做公式里的哪一部分：

### 4.1 创建空矩阵

```python
pe = torch.zeros(max_len, d_model)  # (512, d_model)
```
一个表格：512 行（位置），d_model 列（维度），待填充。

### 4.2 构造公式中的 `pos`

```python
position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
# 结果: [[0.],
#        [1.],
#        [2.],
#        ...
#        [511.]]
# shape: (512, 1)
```

就是公式里的 $pos$，竖着排列，`unsqueeze(1)` 是为了后面能和 `div_term` 做矩阵广播相乘。

### 4.3 构造公式中的除数项 `div_term`

公式里需要 $\frac{1}{10000^{2i/d_{model}}}$，代码用对数变换来算（数值更稳定）：

```python
div_term = torch.exp(
    torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
)
```

拆解这个数学等价关系：

$$\frac{1}{10000^{2i/d}} = e^{\,-2i \cdot \ln(10000)/d}$$

```
原式:  1 / 10000^(2i/d)
     = 10000^(-2i/d)
     = e^(ln(10000) · (-2i/d))
     = e^(-2i · ln(10000) / d)     ← 代码算的就是这个
```

`torch.arange(0, d_model, 2)` 生成 `[0, 2, 4, ..., d_model-2]`，对应所有的 $2i$。

所以 `div_term` 是一个横向量，shape 为 `(d_model/2,)`，值从大到小递减：

```
div_term ≈ [1.0, 0.xx, 0.xx, ..., 0.0001]
             ↑高频(变化快)          ↑低频(变化慢)
```

### 4.4 填充 sin 和 cos

```python
pe[:, 0::2] = torch.sin(position * div_term)   # 偶数列
pe[:, 1::2] = torch.cos(position * div_term)   # 奇数列
```

`position * div_term` 就是 $pos \times \frac{1}{10000^{2i/d}} = \frac{pos}{10000^{2i/d}}$

这就是公式里 sin 和 cos 括号内的东西。

广播机制：`(512,1) × (1, d_model/2)` → `(512, d_model/2)`，每个位置和每个维度的组合都算出一个值。

### 4.5 注册为 buffer

```python
pe = pe.unsqueeze(0)        # (1, 512, d_model)，加 batch 维度
self.register_buffer('pe', pe)
```

`register_buffer` 意思是：**这个张量是模型的一部分，但不需要训练更新**。它会：
- 跟着模型一起 `.cuda()` / `.cpu()`
- 跟着模型一起保存和加载
- **不会**被优化器更新

### 4.6 前向传播：直接相加

```python
def forward(self, x):
    x = x + self.pe[:, :x.size(1), :]   # 截取实际长度，加到词嵌入上
    return self.dropout(x)
```

```
词嵌入（语义信息）  +  位置编码（位置信息）  =  带位置的表示
(batch, seq_len, d_model)                    (batch, seq_len, d_model)
```

---

## 完整逻辑链总结

```
目标：给每个位置一个唯一且有界的向量
  ↓
灵感：模仿二进制——不同列用不同速度变化
  ↓
方法：用不同频率的 sin/cos 代替二进制的 0/1 跳变
  ↓
公式：sin(pos / 分母) 和 cos(pos / 分母)
      分母随维度递增 → 频率递减 → 前面维度变化快，后面维度变化慢
  ↓
代码：
  position = pos 列向量
  div_term = 1/分母 行向量（用 exp+log 计算）
  pe = sin/cos(position × div_term)
  ↓
使用：词嵌入 + pe → 送入 Transformer
```

# 为什么不用二进制编码？

你的直觉很好，这个问题非常值得深究。二进制编码确实有几个致命缺陷。

---

## 缺陷一：相邻位置的编码可能差异巨大

看二进制编码：

```
位置 7:   0  1  1  1
位置 8:   1  0  0  0
              ↑  ↑  ↑  ↑
           4个位全部翻转！
```

7 和 8 明明是**相邻位置**，但编码的差异是**最大的**（所有位都不同）。

而 sin/cos 编码，相邻位置的向量只有微小的平滑变化，位置越近 → 向量越相似。

> 对于语言来说，"相邻的词通常关系更密切"，编码应该反映这一点。

---

## 缺陷二：维度利用效率极低

二进制编码 512 个位置只需要 $\log_2(512) = 9$ 位。

但 `d_model` 通常是 256 或 512，剩下的维度怎么办？

```
位置3 的二进制编码 (d_model=512):
[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, ..., 0, 0, 0]
                       ↑有用↑   ↑----503个0，全浪费----↑
```

sin/cos 编码则**每个维度都携带信息**，充分利用了全部 d_model 个维度。

---

## 缺陷三（最重要）：无法表达相对位置

这是 sin/cos 编码最精妙的性质。

sin/cos 有一个三角恒等式：

$$\sin(pos + k) = \sin(pos)\cos(k) + \cos(pos)\sin(k)$$

这意味着：**位置 pos+k 的编码可以由位置 pos 的编码通过一个线性变换得到**，而且这个变换**只取决于偏移量 k，与 pos 无关**。

$$\begin{bmatrix} PE(pos+k, 2i) \\ PE(pos+k, 2i+1) \end{bmatrix} = \begin{bmatrix} \cos(k \cdot \omega_i) & \sin(k \cdot \omega_i) \\ -\sin(k \cdot \omega_i) & \cos(k \cdot \omega_i) \end{bmatrix} \begin{bmatrix} PE(pos, 2i) \\ PE(pos, 2i+1) \end{bmatrix}$$

**通俗说**：模型可以通过一个固定的矩阵乘法，从"当前位置"跳到"前面第 k 个位置"。这让模型非常容易学会"往前看 3 个词"、"往后看 5 个词"这样的相对位置关系。

二进制编码完全没有这个性质：
```
位置 2:   0 1 0
位置 5:   1 0 1    (偏移+3)

位置 10:  1 0 1 0
位置 13:  1 1 0 1  (同样偏移+3)

两组之间找不到任何统一的变换规则
```

---

## 缺陷四：只有 0 和 1，与神经网络不兼容

神经网络靠**梯度**学习，擅长处理**连续平滑**的值。

- sin/cos 的值在 **[-1, 1]** 之间连续变化 ✅
- 二进制只有 **{0, 1}**，是离散跳变 ❌

离散的输入会让梯度的传播变得困难，模型更难从中提取有用的模式。

---

## 一句话总结

| | 二进制编码 | sin/cos 编码 |
|--|----------|------------|
| 相邻位置相似？ | ❌ 可能完全不同 | ✅ 平滑过渡 |
| 维度利用率 | ❌ 只用 9 维 | ✅ 全部 d_model 维 |
| 能表达相对位置？ | ❌ 无规律 | ✅ 线性变换 |
| 与神经网络兼容？ | ❌ 离散值 | ✅ 连续值 |

**二进制编码提供了正确的直觉（不同列、不同频率），但 sin/cos 是这个直觉在连续空间中的正确实现。**

In [ ]:
class CNN_Transformer(nn.Module):
    """
    架构: CNN(1024→128) → Positional Encoding → Transformer Encoder → GAP → Classifier
    输入: (batch, 1024, 2)
    """
    def __init__(self, num_classes=9, d_model=128, nhead=8,
                 num_layers=4, dim_feedforward=256, dropout=0.3):
        super().__init__()

        self.cnn = CNNFrontend(in_channels=2, out_channels=d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=256, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = ClassifierHead(d_model, num_classes)

    def forward(self, x):
        # x: (batch, 1024, 2)
        x = x.permute(0, 2, 1)  # → (batch, 2, 1024)
        x = self.cnn(x)         # → (batch, 128, 128)
        x = x.permute(0, 2, 1)  # → (batch, 128, 128)  [seq_len=128, d=128]
        x = self.pos_encoder(x) # + 位置编码
        x = self.transformer_encoder(x) # → (batch, 128, 128)
        x = self.final_norm(x)
        x = x.mean(dim=1)       # Global Average Pooling → (batch, 128)
        return self.classifier(x)
    
def build_model(classes=9):
    return CNN_Transformer(num_classes=classes)

# CNN_Transformer 功能块总结

---

## 整体架构一览

```
输入 (batch, 1024, 2)
  ↓
┌─────────────────────────┐
│  ① CNN Frontend         │  压缩序列长度，提取局部特征
│     (1024,2) → (128,128)│
├─────────────────────────┤
│  ② Positional Encoding  │  注入位置信息
│     (128,128) → (128,128)│
├─────────────────────────┤
│  ③ Transformer Encoder  │  建模全局依赖关系
│     (128,128) → (128,128)│
├─────────────────────────┤
│  ④ LayerNorm + GAP      │  归一化 + 全局平均池化
│     (128,128) → (128,)  │
├─────────────────────────┤
│  ⑤ Classifier Head      │  映射到类别
│     (128,) → (num_classes,)│
└─────────────────────────┘
  ↓
输出 (batch, num_classes)
```

---

## ① CNN Frontend —— 局部特征提取与降采样

### 原理
用一维卷积扫描原始序列，提取局部模式（如短时波形特征），同时通过步长或池化将序列长度从 1024 压缩到 128，降低后续 Transformer 的计算量。

### 参数

| 参数 | 含义 | 本例取值 |
|------|------|---------|
| `in_channels` | 输入通道数（特征维度） | 2 |
| `out_channels` | 输出通道数 = `d_model` | 128 |

### 输入输出

```
输入: (batch, 2, 1024)     ← 2个通道，1024个时间步
输出: (batch, 128, 128)    ← 128个通道，128个时间步
```

### 调参指南

| 调什么 | 怎么调 | 影响 |
|--------|--------|------|
| 卷积层数 | 增加层数 → 提取更复杂的局部模式 | 过多会丢失细节 |
| 降采样倍率 | 输出序列越短 → Transformer 越快 | 过短会丢失时序信息 |
| `out_channels` | 必须等于 `d_model` | 决定整个模型的宽度 |

---

## ② Positional Encoding —— 位置信息注入

### 原理
用不同频率的 sin/cos 函数为每个位置生成唯一编码，加到特征向量上，使 Transformer 能感知序列顺序。

### 参数

| 参数 | 含义 | 本例取值 |
|------|------|---------|
| `d_model` | 编码维度，必须与输入特征维度一致 | 128 |
| `max_len` | 支持的最大序列长度 | 256 |
| `dropout` | 随机置零比例 | 0.3 |

### 输入输出

```
输入: (batch, 128, 128)    ← seq_len=128, d_model=128
输出: (batch, 128, 128)    ← 形状不变，值被加上位置信息
```

### 调参指南

| 调什么 | 怎么调 | 影响 |
|--------|--------|------|
| `max_len` | ≥ 实际最大序列长度即可 | 设太小会报错 |
| `dropout` | 0.1~0.3 | 正则化强度，数据少时调大 |
| **一般不需要调** | 这个模块几乎是固定的 | — |

---

## ③ Transformer Encoder —— 全局关系建模

### 原理
通过多头自注意力机制，让序列中**任意两个位置**都能直接交互信息。堆叠多层后，模型能捕获复杂的长距离依赖关系。每层结构：

```
输入 → LayerNorm → 多头自注意力 → 残差连接
    → LayerNorm → 前馈网络(FFN) → 残差连接 → 输出
```

### 参数

| 参数 | 含义 | 本例取值 | 约束 |
|------|------|---------|------|
| `d_model` | 特征维度 | 128 | 必须能被 `nhead` 整除 |
| `nhead` | 注意力头数 | 8 | 每个头维度 = 128/8 = 16 |
| `num_layers` | 堆叠层数 | 4 | — |
| `dim_feedforward` | FFN 隐藏层维度 | 256 | 通常为 `d_model` 的 2~4 倍 |
| `dropout` | 注意力和FFN中的dropout | 0.3 | — |
| `norm_first` | Pre-Norm 还是 Post-Norm | True | Pre-Norm 训练更稳定 |
| `activation` | FFN 激活函数 | gelu | relu 或 gelu |

### 输入输出

```
输入: (batch, 128, 128)    ← 128个token，每个128维
输出: (batch, 128, 128)    ← 形状不变，但每个token融合了全局信息
```

### 调参指南（重点）

| 调什么 | 怎么调 | 影响 |
|--------|--------|------|
| `d_model` | 64 / 128 / 256 / 512 | **最核心参数**，决定模型容量。数据少用小值，数据多用大值 |
| `nhead` | 4 / 8 / 16 | 头数越多，捕获的关系模式越多样。每头维度建议 ≥ 16 |
| `num_layers` | 2 / 4 / 6 / 8 | 层数越多，表达能力越强，但更容易过拟合 |
| `dim_feedforward` | `d_model` × 2~4 | FFN 的容量，对性能有影响但不如前三个敏感 |
| `dropout` | 0.1~0.5 | 数据量小 → 调大；数据量大 → 调小 |

**典型配置参考**：

```
小数据/轻量:  d_model=64,  nhead=4, num_layers=2, dim_ff=128
中等:        d_model=128, nhead=8, num_layers=4, dim_ff=256   ← 本例
大数据/重量:  d_model=256, nhead=8, num_layers=6, dim_ff=512
```

---

## ④ LayerNorm + Global Average Pooling —— 归一化与聚合

### 原理

```
LayerNorm: 对每个样本的特征做归一化，稳定数值
GAP:       对序列维度取平均，将变长序列压缩为固定长度向量
```

```python
x = self.final_norm(x)    # (batch, 128, 128) → (batch, 128, 128)
x = x.mean(dim=1)         # (batch, 128, 128) → (batch, 128)
```

### 为什么用 GAP 而不是只取第一个 token

```
取第一个token:  x[:, 0, :]     ← 信息可能集中不够
GAP:           x.mean(dim=1)  ← 综合所有位置的信息，更鲁棒
```

### 调参指南
**不需要调**，这一步没有可学习参数（LayerNorm 有，但不需要手动调）。

---

## ⑤ Classifier Head —— 分类输出

### 原理
一个或多个全连接层，将特征向量映射到类别数量。

### 参数

| 参数 | 含义 | 本例取值 |
|------|------|---------|
| `d_model` | 输入维度 | 128 |
| `num_classes` | 输出类别数 | 9 |

### 输入输出

```
输入: (batch, 128)
输出: (batch, 9)     ← 每个类别的 logit（未归一化分数）
```

---

## 总体调参优先级

```
影响最大，优先调:
  ❶ d_model        ← 整个模型的宽度
  ❷ num_layers     ← 模型的深度
  ❸ dropout        ← 防过拟合的关键

其次:
  ❹ nhead          ← 注意力的多样性
  ❺ dim_feedforward ← FFN容量

最后（通常不动）:
  ❻ max_len、activation、norm_first
```

**过拟合时**：增大 dropout、减小 d_model、减少 num_layers

**欠拟合时**：增大 d_model、增加 num_layers、减小 dropout